# Шаг 5. Энкодер: перенос на RuATD и размер для браузера

Э2 показал, что дообученный энкодер XLM-RoBERTa точнее гибридного детектора на корпусе M4. Чтобы сравнение с гибридом было честным, энкодер нужно проверить в тех же условиях:
1. **Перенос на другой корпус (как Э3):** энкодеры, обученные на M4, проверяются на 6000 текстах CoAT (RuATD) — тех же, что в Э3. У гибрида там AUROC 0,712, у стилометрии 0,435.
2. **Размер и скорость:** энкодер экспортируется в ONNX, квантуется в INT8, измеряются объём, точность после квантования и время на один текст на процессоре — как для GPT-2 и ruGPT-3 в шаге 3.

Время: ≈30–35 минут. **Перед запуском:** среда **T4 GPU**, затем «Выполнить все».
**Прислать:** вывод ячеек 4–6 и файл `results_step5.json` (папка `ai-text-detector` на Диске).

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, torch
WORK = '/content/drive/MyDrive/ai-text-detector'
os.makedirs('/content/code', exist_ok=True)
os.chdir('/content/code'); sys.path.insert(0, '/content/code')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'Нужна среда с GPU: Среда выполнения → Сменить среду выполнения → T4 GPU'
!pip -q install onnx onnxruntime 2>&1 | tail -1

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _convert(word, table):
    return ''.join(table.get(ch, ch) for ch in word)

def _mixed(word):
    return any('\u0400' <= ch <= '\u04FF' for ch in word) and any('a' <= ch.lower() <= 'z' for ch in word)

def _fix_word(word, doc_script):
    """Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно
    (в слове есть буквы без двойников) — к алфавиту большинства букв слова (защита от гомоглифов)."""
    if not _mixed(word):
        # слово целиком из «чужого» алфавита, но только из букв-двойников (например, кириллическая «а» в английском тексте)
        foreign = CYR2LAT if doc_script == 'lat' else LAT2CYR
        if word and all(ch in foreign for ch in word):
            return _convert(word, foreign)
        return word
    first = _convert(word, LAT2CYR if doc_script == 'cyr' else CYR2LAT)
    if not _mixed(first):
        return first
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    return _convert(word, LAT2CYR if cyr > lat else CYR2LAT)

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    doc_script = 'cyr' if cyr_ratio(t) > 0.5 else 'lat'
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0), doc_script), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }

In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res

In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res

In [ ]:
%%writefile /content/code/data_loading.py
"""Поиск и загрузка данных M4 и RuATD в единую таблицу:
text | label (1 = ИИ, 0 = человек) | lang | source | generator | pair_id"""
import os, json, glob, hashlib
import pandas as pd
from stylometry import detect_lang

HUMAN_KEYS = ['human_text', 'human', 'text_human', 'original_text']
MACHINE_KEYS = ['machine_text', 'machine', 'text_machine', 'generated_text', 'ai_text']
TEXT_KEYS = ['text', 'Text', 'content']
LABEL_KEYS = ['label', 'Label', 'Class', 'class', 'labels']
MODEL_KEYS = ['model', 'Model', 'generator', 'source_model']

def _first(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ''):
            return k
    return None

def _read_records(path, limit=None):
    recs = []
    with open(path, encoding='utf-8') as f:
        head = f.read(1); f.seek(0)
        if head == '[':                       # обычный JSON-массив
            data = json.load(f); recs = data[:limit] if limit else data
        else:                                 # JSON Lines
            for line in f:
                line = line.strip()
                if not line: continue
                try: recs.append(json.loads(line))
                except json.JSONDecodeError: continue
                if limit and len(recs) >= limit: break
    return recs

def _pid(*parts):
    return hashlib.md5('|'.join(map(str, parts)).encode()).hexdigest()[:12]

def _label_value(v):
    s = str(v).strip().lower()
    if s in ('1', 'm', 'machine', 'ai', 'generated', 'fake', 'true'): return 1
    if s in ('0', 'h', 'human', 'real', 'false'): return 0
    return None

def inspect_files(root, exts=('.jsonl', '.json', '.csv', '.tsv'), max_files=200):
    """Печатает список файлов данных и ключи первой записи — для диагностики."""
    files = [p for p in glob.glob(os.path.join(root, '**', '*'), recursive=True)
             if p.lower().endswith(exts) and os.path.getsize(p) > 0]
    print(f'{root}: найдено файлов данных — {len(files)}')
    for p in sorted(files)[:max_files]:
        size = os.path.getsize(p) / 1e6
        try:
            if p.lower().endswith(('.csv', '.tsv')):
                cols = list(pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',', nrows=2).columns)
            else:
                r = _read_records(p, limit=1); cols = list(r[0].keys()) if r else []
        except Exception as e:
            cols = [f'ошибка чтения: {e}']
        print(f'  {os.path.relpath(p, root)}  ({size:.1f} МБ)  поля: {cols[:12]}')
    return files

def load_json_dir(root, dataset_name, limit_per_file=None):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.json*'), recursive=True):
        try: recs = _read_records(p, limit=limit_per_file)
        except Exception: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, r in enumerate(recs):
            if not isinstance(r, dict): continue
            gen = r.get(_first(r, MODEL_KEYS)) if _first(r, MODEL_KEYS) else fname
            hk, mk = _first(r, HUMAN_KEYS), _first(r, MACHINE_KEYS)
            if hk or mk:                                   # параллельный формат (как в M4)
                pid = _pid(dataset_name, r[hk]) if hk else _pid(dataset_name, fname, i)  # пара = один человеческий текст
                if hk: rows.append((r[hk], 0, fname, 'human', pid))
                if mk:
                    m = r[mk]
                    for j, t in enumerate(m if isinstance(m, list) else [m]):
                        rows.append((t, 1, fname, str(gen), pid))
                continue
            tk, lk = _first(r, TEXT_KEYS), _first(r, LABEL_KEYS)
            if tk and lk is not None and _label_value(r[lk]) is not None:
                rows.append((r[tk], _label_value(r[lk]), fname, str(gen) if _label_value(r[lk]) else 'human', _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

def load_csv_dir(root, dataset_name):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.[ct]sv'), recursive=True):
        try: df = pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',')
        except Exception: continue
        tk = _first({c: 1 for c in df.columns}, TEXT_KEYS)
        lk = _first({c: 1 for c in df.columns}, LABEL_KEYS)
        if not tk or not lk: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, (t, l) in enumerate(zip(df[tk], df[lk])):
            lab = _label_value(l)
            if lab is None:                  # многоклассовая разметка: 'Human' = 0, имя модели = 1
                lab = 0 if str(l).strip().lower() == 'human' else 1
                gen = 'human' if lab == 0 else str(l)
            else:
                gen = 'human' if lab == 0 else 'unknown'
            rows.append((t, lab, fname, gen, _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

# Язык M4 определяем по имени файла: кириллицей пишутся и болгарский, и казахский,
# латиницей — немецкий и индонезийский, поэтому автоопределение по алфавиту здесь ненадёжно.
M4_LANG_BY_PREFIX = {'russian': 'ru', 'arxiv': 'en', 'peerread': 'en', 'reddit': 'en', 'wikihow': 'en', 'wikipedia': 'en'}
BLOOMZ_HUMAN_KEYS = {'arxiv': 'abstract', 'wikihow': 'text', 'wikipedia': 'text', 'reddit': 'text'}

def load_m4(root):
    """Загрузка английской и русской частей M4 с учётом разных форматов файлов."""
    rows = []
    for p in sorted(glob.glob(os.path.join(root, 'data', '*.jsonl'))):
        fname = os.path.splitext(os.path.basename(p))[0]
        prefix, gen_from_name = fname.split('_')[0], fname.split('_')[-1]
        lang = M4_LANG_BY_PREFIX.get(prefix)
        if lang is None:
            continue
        for i, r in enumerate(_read_records(p)):
            gen = str(r.get('model') or gen_from_name)
            human = r.get('human_text') or (r.get(BLOOMZ_HUMAN_KEYS.get(prefix, '')) if 'bloomz' in fname else None)
            machine = r.get('machine_text')
            if prefix == 'peerread' and 'bloomz' in fname:          # отзывы хранятся списками
                human = (r.get('human_reviews') or [None])[0]
                machine = (r.get('bloom_reviews') or [None])[0]
                gen = 'bloomz'
            pid = _pid('M4', human) if isinstance(human, str) else _pid('M4', fname, i)
            if isinstance(human, str): rows.append((human, 0, fname, 'human', pid, lang))
            for t in (machine if isinstance(machine, list) else [machine]):
                if isinstance(t, str): rows.append((t, 1, fname, gen, pid, lang))
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id', 'lang'])
    df = df[df.text.str.strip().str.len() > 0]
    df['label'] = df.label.astype(int); df['dataset'] = 'M4'
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def load_coat(config='binary'):
    """Корпус CoAT (опубликованная версия RuATD) с Hugging Face. Столбцы определяются автоматически."""
    from datasets import load_dataset
    ds = load_dataset('RussianNLP/coat', config)
    print(ds)
    frames = []
    for split in ds:
        d = ds[split].to_pandas()
        feats = ds[split].features
        text_col = max([c for c in d.columns if d[c].dtype == object or pd.api.types.is_string_dtype(d[c])],
                       key=lambda c: d[c].astype(str).str.len().mean())
        label_col = None
        for c in d.columns:
            if c == text_col: continue
            names = getattr(feats[c], 'names', None)
            vals = names or sorted(map(str, d[c].dropna().unique()))[:50]
            if any(str(v).lower() in ('h', 'human', 'm', 'machine') for v in vals):
                label_col = c; break
            if c == 'label' and set(map(str, d[c].dropna().unique())) <= {'0', '1'}:
                label_col = c; break    # карточка CoAT: 1 — машинный текст, 0 — человеческий
        if label_col is None:
            print(f'[{split}] столбец с меткой не найден, столбцы: {list(d.columns)}'); continue
        names = getattr(feats[label_col], 'names', None)
        def to_name(v):
            if names and str(v).isdigit() and int(v) < len(names): return names[int(v)]
            return v
        lab = d[label_col].map(to_name)
        lab = lab.astype(str).str.lower()
        if lab.isin(['0', '1']).all():
            y = (lab == '1').astype(int); gen = ['human' if v == 0 else 'unknown' for v in y]
        elif lab.isin(['h', 'm', 'human', 'machine']).all():
            y = lab.isin(['m', 'machine']).astype(int); gen = ['human' if v == 0 else 'unknown' for v in y]
        else:                                     # многоклассовая разметка: human или имя модели
            y = (lab != 'human').astype(int); gen = lab.tolist()
        frames.append(pd.DataFrame({'text': d[text_col], 'label': y, 'source': f'coat_{split}', 'generator': gen,
                                    'pair_id': [_pid('coat', split, i) for i in range(len(d))], 'lang': 'ru'}))
        print(f'[{split}] текст: {text_col}, метка: {label_col}, примеры меток: {lab.unique()[:8]}')
    if not frames: return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True); df['dataset'] = 'RuATD'
    df = df[df.text.astype(str).str.strip().str.len() > 0]
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def _finish(rows, dataset_name):
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id'])
    if df.empty: return df
    df = df[df.text.apply(lambda t: isinstance(t, str) and len(t.strip()) > 0)].copy()
    df['lang'] = df.text.apply(detect_lang)
    df['label'] = df.label.astype(int)
    df['dataset'] = dataset_name
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

In [ ]:
%%writefile /content/code/encoder.py
"""Дообучение многоязычного энкодера (XLM-RoBERTa) для обнаружения сгенерированного текста (серия Э2).
Обычный цикл обучения PyTorch: так меньше зависимости от версий библиотек, чем с Trainer."""
import math, time, random
import numpy as np


def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def encode(tok, texts, max_len=256):
    """Токенизация один раз заранее: список списков идентификаторов токенов (без дополнения)."""
    return tok(list(texts), truncation=True, max_length=max_len, add_special_tokens=True)['input_ids']


def _batch(ids_list, pad_id, device):
    import torch
    L = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), L), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x, dtype=torch.long); mask[i, :len(x)] = 1
    return ids.to(device), mask.to(device)


def train(init, ids, labels, pad_id, steps=None, epochs=None, batch_size=16, lr=2e-5, warmup=0.1,
          weight_decay=0.01, seed=0, device='cuda', log_every=100):
    """Дообучает классификатор. init — имя модели на Hugging Face или путь к сохранённой модели.
    Длительность задаётся либо числом шагов (steps), либо числом эпох (epochs)."""
    import torch
    from transformers import AutoModelForSequenceClassification
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(init, num_labels=2).to(device)
    labels = np.asarray(labels, dtype=np.int64); n = len(labels)
    per_epoch = math.ceil(n / batch_size)
    total = steps if steps is not None else per_epoch * epochs
    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    groups = [{'params': [p for nm, p in model.named_parameters() if not any(k in nm for k in no_decay)], 'weight_decay': weight_decay},
              {'params': [p for nm, p in model.named_parameters() if any(k in nm for k in no_decay)], 'weight_decay': 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    n_warm = max(1, int(total * warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / n_warm if s < n_warm else max(0.0, (total - s) / max(1, total - n_warm)))
    use_amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    rng = np.random.default_rng(seed)
    model.train(); step = 0; t0 = time.time(); run_loss = []
    while step < total:
        order = rng.permutation(n)                      # новая перестановка на каждую эпоху
        for b in range(0, n, batch_size):
            if step >= total: break
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            y = torch.tensor(labels[idx], device=device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss.append(loss.item()); step += 1
            if log_every and step % log_every == 0:
                print(f'    шаг {step}/{total}  loss={np.mean(run_loss[-log_every:]):.3f}  {time.time() - t0:.0f} с')
    model.eval()
    return model


def predict(model, ids, pad_id, batch_size=64, device='cuda'):
    """Вероятность класса «сгенерирован» (метка 1) для каждого текста."""
    import torch
    out = []
    order = np.argsort([len(x) for x in ids])          # близкие по длине тексты в одном пакете — быстрее
    with torch.no_grad():
        for b in range(0, len(ids), batch_size):
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                logits = model(input_ids=x, attention_mask=m).logits.float()
            out.append((idx, torch.softmax(logits, -1)[:, 1].cpu().numpy()))
    p = np.zeros(len(ids), dtype=np.float64)
    for idx, pr in out: p[idx] = pr
    return p


def free():
    """Освобождает память видеокарты. Перед вызовом удалите модель: del model."""
    import torch, gc
    gc.collect(); torch.cuda.empty_cache()


def few_shot_sample(labels, n, seed):
    """Индексы n примеров (по n/2 каждого класса) из обучающей выборки."""
    rng = np.random.default_rng(seed); labels = np.asarray(labels)
    pos = rng.choice(np.where(labels == 1)[0], n // 2, replace=False)
    neg = rng.choice(np.where(labels == 0)[0], n - n // 2, replace=False)
    return rng.permutation(np.concatenate([pos, neg]))

In [ ]:
%%writefile /content/code/onnx_utils.py
"""Экспорт языковых моделей в ONNX, квантование INT8 и вычисление признаков через ONNX Runtime (разделы 1.5, 4.1)."""
import os, time, numpy as np

def export_onnx(model, tok, out_dir, opset=17):
    """Экспорт модели (вход: input_ids, attention_mask; выход: logits) и токенизатора в формат для Transformers.js."""
    import torch
    os.makedirs(os.path.join(out_dir, 'onnx'), exist_ok=True)
    model = model.float().cpu().eval()
    model.config.use_cache = False

    class Wrap(torch.nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, input_ids, attention_mask):
            return self.m(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits

    ids = torch.ones(1, 8, dtype=torch.long); mask = torch.ones(1, 8, dtype=torch.long)
    path = os.path.join(out_dir, 'onnx', 'model.onnx')
    kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['logits'], opset_version=opset,
              dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                            'logits': {0: 'batch', 1: 'seq'}})
    try:
        torch.onnx.export(Wrap(model), (ids, mask), path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(Wrap(model), (ids, mask), path, **kw)
    model.config.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)          # быстрый токенизатор сохраняет tokenizer.json
    return path

def last_matmul_name(path):
    """Имя последнего умножения матриц — выходной слой модели (проекция на словарь)."""
    import onnx
    m = onnx.load(path, load_external_data=False)
    return [n.name for n in m.graph.node if n.op_type in ('MatMul', 'Gemm')][-1]

QUANT_VARIANTS = {
    'q8': {},                                         # по умолчанию: один масштаб на всю матрицу весов
    'q8_per_channel': {'per_channel': True},          # свой масштаб для каждого столбца
    'q8_keep_head': {'keep_head': True},              # выходной слой остаётся в fp32
    'q8_pc_keep_head': {'per_channel': True, 'keep_head': True},
}

def quantize(path_fp32, variant='q8'):
    from onnxruntime.quantization import quantize_dynamic, QuantType
    opts = QUANT_VARIANTS[variant]
    out = path_fp32.replace('model.onnx', f'model_{variant}.onnx')
    kw = {'per_channel': opts.get('per_channel', False)}
    if opts.get('keep_head'):
        kw['nodes_to_exclude'] = [last_matmul_name(path_fp32)]
    quantize_dynamic(path_fp32, out, weight_type=QuantType.QInt8, **kw)
    return out

def size_mb(path):
    return os.path.getsize(path) / 2**20

def ort_token_stats(session, tok, texts, max_tokens=256):
    """То же, что lm_features.token_stats, но через ONNX Runtime на CPU — по одному тексту, как в браузере."""
    out, times = [], []
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True, max_length=max_tokens, return_tensors='np')['input_ids'].astype(np.int64)
        if ids.shape[1] < 2:
            out.append({'ids': ids[0], 'lp': np.array([]), 'ent': np.array([]), 'var': np.array([]), 'rank': np.array([])}); continue
        t0 = time.perf_counter()
        logits = session.run(['logits'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0, :-1].astype(np.float64)
        times.append(time.perf_counter() - t0)
        mx = logits.max(1, keepdims=True)
        logp = logits - (np.log(np.exp(logits - mx).sum(1, keepdims=True)) + mx)
        tgt = ids[0, 1:]
        lp = logp[np.arange(len(tgt)), tgt]
        p = np.exp(logp)
        ent = -(p * logp).sum(1)
        var = (p * logp ** 2).sum(1) - ent ** 2
        rank = (logp > lp[:, None]).sum(1) + 1
        out.append({'ids': ids[0], 'lp': lp, 'ent': ent, 'var': var, 'rank': rank})
    return out, times

## 3. Данные: M4 (те же выборки) и CoAT (те же 6000 текстов, что в Э3)

In [ ]:
import importlib, time, json, numpy as np, pandas as pd
import stylometry, metrics_utils, baseline_stylometry, data_loading, encoder, onnx_utils
for m in (stylometry, metrics_utils, baseline_stylometry, data_loading, encoder, onnx_utils): importlib.reload(m)
from stylometry import normalize
from baseline_stylometry import split
from data_loading import load_coat
from metrics_utils import evaluate
from encoder import encode, train, predict, free
SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet'); df = df[df.dataset == 'M4']
def bal(d, cap): return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {'en': bal(df[df.lang == 'en'], 6000), 'ru': df[df.lang == 'ru']}
coat = load_coat('binary'); coat['n_words'] = coat.text.astype(str).str.split().str.len()
data['coat'] = bal(coat[coat.n_words >= 30], 3000)
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
SPL, Y = {}, {}
for lang in ['en', 'ru']:
    tr, va, te = split(data[lang], SEED)
    SPL[lang] = {'tr': tr.index.values, 'va': va.index.values, 'te': te.index.values}
Y = {k: d.label.values.astype(int) for k, d in data.items()}
assert len(SPL['ru']['te']) == 1350 and len(SPL['en']['te']) == 1803, 'Разбиение не совпало — напиши мне'
print('CoAT:', len(data['coat']), 'текстов, из них ИИ:', int(Y['coat'].sum()))
RES = {}
def save(): json.dump(RES, open(f'{WORK}/results_step5.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
def short(r): return f"AUROC={r['AUROC']:.3f} F1={r['F1']:.3f} FPR={r['FPR']:.3f} TPR@1%={r['TPR@1%FPR']:.3f}"
from transformers import AutoTokenizer
ENC = 'FacebookAI/xlm-roberta-base'
tok = AutoTokenizer.from_pretrained(ENC); PAD = tok.pad_token_id
IDS = {k: encode(tok, d.norm, max_len=256) for k, d in data.items()}
take = lambda k, part=None: IDS[k] if part is None else [IDS[k][i] for i in SPL[k][part]]
y_of = lambda k, part=None: Y[k] if part is None else Y[k][SPL[k][part]]

## 4. Энкодеры, обученные на M4, на корпусе CoAT
Три энкодера (как в Э2): только английский; только все русские; английский, затем все русские. Каждый проверяется на русском тесте M4 и на CoAT. Около 15 минут.

In [ ]:
RES['coat_transfer'] = {}
def run(name, init, k, part, epochs, save_to=None):
    t0 = time.time()
    m = train(init, take(k, part), y_of(k, part), PAD, epochs=epochs, seed=0, log_every=0)
    r = {'ru_test': evaluate(y_of('ru', 'te'), predict(m, take('ru', 'te'), PAD)),
         'coat': evaluate(y_of('coat'), predict(m, take('coat'), PAD))}
    if save_to: m.save_pretrained(save_to); tok.save_pretrained(save_to)
    del m; free()
    RES['coat_transfer'][name] = r; save()
    print(f'{name:28s} ({time.time() - t0:.0f} с)\n   русский тест M4: {short(r["ru_test"])}\n   CoAT:            {short(r["coat"])}')
run('только английский', ENC, 'en', 'tr', 2, save_to='/content/enc_en')
run('только русские (все)', ENC, 'ru', 'tr', 2)
run('англ. → + русские (все)', '/content/enc_en', 'ru', 'tr', 2, save_to='/content/enc_enru')

## 5. Энкодер в формате для браузера: ONNX, INT8, время на процессоре
Экспортируется лучший энкодер («англ. → + русские»). Точность сравнивается на 300 текстах русского теста (150 + 150), время — при подаче текстов по одному, как в шаге 3.

In [ ]:
import onnxruntime as ort
from transformers import AutoModelForSequenceClassification
from onnx_utils import quantize, size_mb
out_dir = '/content/enc_onnx'; os.makedirs(f'{out_dir}/onnx', exist_ok=True)
m = AutoModelForSequenceClassification.from_pretrained('/content/enc_enru').float().cpu().eval()
class Wrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask): return self.m(input_ids=input_ids, attention_mask=attention_mask).logits
x = torch.ones(1, 8, dtype=torch.long)
kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['logits'], opset_version=17,
          dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'}, 'logits': {0: 'batch'}})
fp32 = f'{out_dir}/onnx/model.onnx'
try: torch.onnx.export(Wrap(m), (x, x), fp32, dynamo=False, **kw)
except TypeError: torch.onnx.export(Wrap(m), (x, x), fp32, **kw)
del m
P = {'fp32': fp32, 'q8': quantize(fp32, 'q8'), 'q8_per_channel': quantize(fp32, 'q8_per_channel')}
RES['onnx'] = {k: {'size_mb': round(size_mb(p), 1)} for k, p in P.items()}
print('Объём, МБ:', {k: v['size_mb'] for k, v in RES['onnx'].items()})
te = SPL['ru']['te']; rng = np.random.default_rng(SEED)
sub = np.concatenate([rng.choice(te[Y['ru'][te] == c], 150, replace=False) for c in (0, 1)])
for kind, path in P.items():
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    probs, times = [], []
    for i in sub:
        ids = np.array([IDS['ru'][i]], dtype=np.int64)
        t0 = time.perf_counter()
        lg = sess.run(['logits'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0]
        times.append(time.perf_counter() - t0)
        e = np.exp(lg - lg.max()); probs.append(e[1] / e.sum())
    r = evaluate(Y['ru'][sub], np.array(probs), n_boot=0)
    r['ms_per_text_median'] = float(np.median(times) * 1000); r['ms_per_text_p95'] = float(np.percentile(times, 95) * 1000)
    RES['onnx'][kind].update(r); save(); del sess
    print(f'{kind:15s} {RES["onnx"][kind]["size_mb"]:7.1f} МБ  AUROC={r["AUROC"]:.3f} F1={r["F1"]:.3f} | медиана {r["ms_per_text_median"]:.0f} мс, 95% {r["ms_per_text_p95"]:.0f} мс')

## 6. Итог

In [ ]:
print(json.dumps({k: {kk: round(vv['AUROC'], 3) for kk, vv in v.items()} for k, v in RES['coat_transfer'].items()}, ensure_ascii=False, indent=1))
print({k: (v['size_mb'], round(v['AUROC'], 3), round(v['ms_per_text_median'])) for k, v in RES['onnx'].items()})
print('Сохранено: results_step5.json')